# 2.5 The judge: rubric v1 on two fixtures

Follows [05-judge-loop.md](../05-judge-loop.md). Edit `assets/rubric.yaml`, then judge a
keyframe that breaks the brief on purpose and one that meets it. `review_loop.py` puts this
judge inside the full loop.

`MODE = "fake"`: scripted judge replies, no network. `"live"`: the judge model on Token Factory.

In [ ]:
MODE = "fake"  # "fake" or "live"

In [ ]:
import sys
from pathlib import Path

MODULE = "02-multi-agent-media-workflows"
LESSON = "05-judge-loop"  # runs go to code/runs/05-judge-loop/<run_id>/
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
CODE = next(p for p in candidates if (p / "tools.py").exists())
sys.path.insert(0, str(CODE))
ASSETS = CODE.parent / "assets"

In [ ]:
import json

from pydantic_ai import Agent

from fake import block_real_models, scripted_model
from models import JUDGE, Brief
from review_loop import Rubric, judge_keyframe, judge_reply, read_png
from tools import llm_model
from tracing import Tracer, new_run_id

brief = Brief.from_yaml(ASSETS / "brief.yaml")
JUDGE_PRICE = (
    None  # LLMPrice(input_per_m=..., output_per_m=...) from the Token Factory pricing page
)

## Rubric v1

Each criterion reads a field of the brief. Keyframe criteria go to the judge; video-only ones
you check by watching the video.

In [ ]:
rubric = Rubric.from_yaml(ASSETS / "rubric.yaml")
print("version", rubric.version)
for c in rubric.criteria:
    print(f"{c.id:10} reads {c.brief_field:10} must_have={c.must_have!s:5} scope={c.scope}")

## Two fixtures

A bad keyframe (expected `RETRY`) and a good one (expected `PASS`), each with a label file.

In [ ]:
from IPython.display import Image, display

tracer = Tracer(new_run_id(), LESSON)
fixtures = []
for name in ("bad_keyframe", "good_keyframe"):
    label = json.loads((ASSETS / "fixtures" / f"{name}.json").read_text())
    png = tracer.folder / f"{name}.png"
    png.write_bytes(read_png(ASSETS / "fixtures" / f"{name}.png", fake=MODE == "fake"))
    fixtures.append((png, label))
    print(name, "expected:", label["expected"], "-", label["why"])
    display(Image(filename=png, width=320))

## Judge both

In [ ]:
if MODE == "fake":
    block_real_models()
    model = scripted_model(
        judge_reply(rubric, 0, suffix="the lamp is in frame"), judge_reply(rubric, 1)
    )
else:
    model = llm_model(JUDGE)
judge = Agent(model, output_type=str)

for png, label in fixtures:
    verdict = await judge_keyframe(
        judge, rubric, brief, png, tracer=tracer, attempt=None, price=JUDGE_PRICE
    )
    got = verdict.verdict if verdict else "unparseable"
    print(f"{png.stem}: expected {label['expected']}, got {got}")
    if verdict:
        print("  reasons:", verdict.reasons, "| fix:", verdict.fix.model_dump())

## Judge cost per verdict

In [ ]:
for e in tracer.events():
    if e.kind == "judge":
        print(
            f"{e.verdict}  tokens {e.tokens_in}+{e.tokens_out}  ${e.cost_usd:.5f}  rubric {e.rubric_version}"
        )